In [90]:
from os import read

from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer
import ast
import pandas as pd

after watching part of the sci kit learn crash course, i realized that i need to do some data processing in order for my ml algos to work. in this case, i need to clean up how my features are represented, and use one hot encoding for the majority of the categorical variables. i also need to scale my values.


In [91]:
data_raw = pd.read_csv("data/ranges.csv")
scaler = StandardScaler() # decided that standard scaler should be ok for this data set.
numeric_collumns = ["width_inches", "height_inches", "depth_inches", "burners", "price", "rating"]
data_raw_scalable_values = data_raw[numeric_collumns]
data_raw_scaled_values = scaler.fit_transform(data_raw_scalable_values)
numeric_data_scaled = pd.DataFrame(data_raw_scaled_values, columns=numeric_collumns)
numeric_data_scaled.to_csv("data/numeric_data_scaled.csv", index=True)



In [92]:
#features will be done themselves, as each row is a array. will be using multi label binarizer from sci kit learn for this.

data_raw["features"] = data_raw["features"].apply(ast.literal_eval) # made the mistake of storing the features as a string instead of a list, easier to do this rather than remake the data set.

mlb = MultiLabelBinarizer()

features_encoded = mlb.fit_transform(data_raw["features"])

features_df = pd.DataFrame(
    features_encoded,
    columns=mlb.classes_,
    index=data_raw.index
)
features_df.to_csv("data/features_encoded.csv", index=True)



In [93]:

encoder_collumns = ["brand", "category", "range_type", "finish"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

encoded_values = encoder.fit_transform(data_raw[encoder_collumns]) #learns what features exist
encoded_names = encoder.get_feature_names_out(encoder_collumns) #gets the feature names out
encoded_df = pd.DataFrame(encoded_values, columns=encoded_names)
encoded_df.to_csv("data/encoded_categorical_values.csv", index=True)



In [94]:
# brand tier encoding

encoder = OrdinalEncoder(categories=[['budget', 'mainstream', 'premium', 'luxury']])
data_raw["brand_tier"] = encoder.fit_transform(data_raw[["brand_tier"]])
brand_tier_encoded = pd.DataFrame(data_raw[["brand_tier"]])
brand_tier_encoded.to_csv("data/brand_tier_encoded.csv", index=True)



# Final dataset creation


In [103]:
# i know that this is technically redundant, but i wanted it to be as reliable as possible.
brand_tier_encoded = pd.read_csv("data/brand_tier_encoded.csv")
encoded_categorical_values = pd.read_csv("data/encoded_categorical_values.csv")
numeric_data_scaled = pd.read_csv("data/numeric_data_scaled.csv")
features_encoded = pd.read_csv("data/features_encoded.csv")

final_data = pd.concat([brand_tier_encoded, encoded_categorical_values, numeric_data_scaled, features_encoded], axis=1)
final_data.drop("Unnamed: 0", axis=1, inplace=True)
final_data.to_csv("data/final_data.csv", index=True)